# Modele na urządzeniach edge, GPU monitoring, Netron, TFLite/LiteRT i TensorFlow.js

Noteboook prezentuje różne środowiska uruchomieniowe dla modeli ML: GPU, urządzenia edge, IoT oraz uruchamianie w przeglądarce.


In [5]:
import tensorflow as tf
from pathlib import Path

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:5000], y_train[:5000], epochs=1, batch_size=128)

Path("models").mkdir(exist_ok=True)
model.save("models/mnist_small.keras")
print("Saved Keras model: models/mnist_small.keras")


40/40 [==============================] - 1s 3ms/step - loss: 1.6334 - accuracy: 0.5648
Saved Keras model: models/mnist_small.keras


## TensorFlow Lite

In [2]:
# Konwersja do TensorFlow Lite / LiteRT
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("models/mnist_small.tflite", "wb") as f:
    f.write(tflite_model)

with open("models/mnist_small.keras", "rb") as f:
    model_bytes = f.read()

print("TF model size:", len(model_bytes) / 1024, "KB")
print("TFLite model size:", len(tflite_model) / 1024, "KB")


INFO:tensorflow:Assets written to: /tmp/tmpu6a3mz9a/assets


INFO:tensorflow:Assets written to: /tmp/tmpu6a3mz9a/assets
W0000 00:00:1781899920.638004     925 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781899920.638019     925 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781899920.638242     925 reader.cc:83] Reading SavedModel from: /tmp/tmpu6a3mz9a
I0000 00:00:1781899920.638826     925 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781899920.638834     925 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpu6a3mz9a
I0000 00:00:1781899920.643746     925 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781899920.644356     925 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781899920.671205     925 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpu6a3mz9a
I0000 00:00:1781899920.677829     925 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Too

TF model size: 321.2060546875 KB
TFLite model size: 101.34375 KB


## Netron

Netron pozwala szybko obejrzeć architekturę modelu.

Setup:
1. Wejść na `https://netron.app` i przeciągnąć plik `models/mnist_small.keras`.
2. Zainstalować lokalnie:

```bash
pip install netron
netron models/mnist_small.keras
```

## nvidia-smi — monitoring GPU

Podstawowe komendy do uruchomienia lokalnie:

```bash
nvidia-smi
nvidia-smi --loop=1
nvidia-smi dmon
nvidia-smi pmon
```


Jakie zasoby mozna monitorować w nvidia-smi?
- wykorzystanie GPU;
- użycie pamięci;
- procesy korzystające z GPU;
- temperatura i pobór mocy;
- czy model faktycznie używa GPU;
- czy bottleneck jest w GPU, CPU, dysku, dataloaderze czy batch size.


In [3]:
# Sprawdzenie GPU z poziomu TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## TensorFlow.js

TensorFlow.js pozwala uruchamiać modele w przeglądarce lub Node.js.

Eksport można wykonać narzędziem `tensorflowjs_converter`:

```bash
pip install tensorflowjs
tensorflowjs_converter --input_format=keras models/mnist_small.keras tfjs_model/
```

W przeglądarce model można załadować przez `tf.loadLayersModel(...)`.


## Podsumowanie

- TFLite/LiteRT jest używane dla edge i on-device inference: telefon, IoT, embedded, urządzenia przemysłowe.
- TensorFlow.js jest przydatne, gdy model ma działać bezpośrednio w przeglądarce.
- Netron pomaga zrozumieć strukturę modelu przed deploymentem.
- `nvidia-smi` jest najprostszym narzędziem do szybkiego sprawdzenia, czy model naprawdę korzysta z GPU i ile zasobów zużywa.
